In [17]:
import json

with open("../schema/schema2.json", "r", encoding="utf-8") as f:
    schema = json.load(f)

In [18]:
def build_search_text(table):
    sections = []
    # Table Information

    sections.append(f"Table:\n{table['table']}")
    # Optional high-level entity
    entity = table.get("entity")
    if entity:
        sections.append(f"Entity:\n{entity}")

    if table.get("description"):
        sections.append(f"Description:\n{table['description']}")

    if table.get("category"):
        sections.append(f"Category:\n{table['category']}")

    # Aliases
    aliases = table.get("aliases", [])
    if aliases:
        sections.append(
            "Aliases:\n" +
            "\n".join(aliases)
        )

    # Business Keywords
    keywords = table.get("business_keywords", [])
    if keywords:
        sections.append(
            "Business Keywords:\n" +
            "\n".join(keywords)
        )

    # Business Concepts
    concepts = table.get("business_concepts", [])
    if concepts:
        sections.append(
            "Business Concepts:\n" +
            "\n".join(concepts)
        )


    # Columns
    column_lines = []

    for col in table.get("columns", []):

        column_lines.append(
            f"{col['name']} ({col['type']})"
        )

        if col.get("description"):
            column_lines.append(col["description"])

        samples = col.get("sample_values", [])
        if samples:
            column_lines.append("Sample Values:")
            column_lines.extend(samples)

        column_lines.append("")

    if column_lines:
        sections.append(
            "Columns:\n" +
            "\n".join(column_lines)
        )

    # Primary Keys
    primary_keys = table.get("primary_keys", [])

    if primary_keys:
        sections.append(
            "Primary Keys:\n" +
            "\n".join(primary_keys)
        )

    # Foreign Keys
    foreign_key_lines = []

    foreign_keys = table.get("foreign_keys", [])

    if foreign_keys:

        for fk in foreign_keys:

            foreign_key_lines.append(
                f"{fk['column']} -> "
                f"{fk['references_table']}."
                f"{fk['references_column']}"
            )

        sections.append(
            "Foreign Keys:\n" +
            "\n".join(foreign_key_lines)
        )

    else:

        sections.append("Foreign Keys:\nNone")

    # ----------------------------
    # Relationships
    # ----------------------------
    relationship_lines = []

    relationships = table.get("relationships", [])

    if relationships:

        for rel in relationships:

            relationship_lines.append(
                f"Related Table: {rel['target_table']}"
            )

            relationship_lines.append(
                f"Relationship Type: {rel['type']}"
            )

            if rel.get("foreign_key"):
                relationship_lines.append(
                    f"Foreign Key: {rel['foreign_key']}"
                )

            if rel.get("description"):
                relationship_lines.append(
                    f"Description: {rel['description']}"
                )

            relationship_lines.append("")

        sections.append(
            "Relationships:\n" +
            "\n".join(relationship_lines)
        )

    # ----------------------------
    # Connected Tables (Optional)
    # ----------------------------
    related_tables = {
        rel["target_table"]
        for rel in relationships
    }

    if related_tables:
        sections.append(
            "Connected Tables:\n" +
            "\n".join(sorted(related_tables))
        )

    return "\n\n".join(sections)

In [19]:
documents = []
table_names = []

for table in schema:
    documents.append(build_search_text(table))
    table_names.append(table["table"])

In [20]:
documents

['Table:\ncustomer\n\nCategory:\nUser Management\n\nAliases:\nuser\nbuyer\nshopper\nclient\naccount holder\nmember\n\nBusiness Keywords:\nuser\nprofile\naccount\ncontact\nemail\nphone\nshopper\nmember\n\nBusiness Concepts:\nCustomer\nIdentity\nAccount Management\nUser Profile\n\nColumns:\ncustomer_id (INTEGER)\nUnique internal identifier for the registered user\n\ncust_code (VARCHAR)\nPublic or external reference code for the client\n\nfull_name (VARCHAR)\nComplete legal or display name of the shopper\n\nemail (VARCHAR)\nPrimary email address used for login and contact\n\nphone (VARCHAR)\nPrimary mobile or landline number for the user\n\nstatus (VARCHAR)\nCurrent state of the account like active, suspended, or closed\n\ncreated_at (TIMESTAMP)\nDate and time when the user registered on the platform\n\n\nPrimary Keys:\ncustomer_id\n\nForeign Keys:\nNone',
 'Table:\ncust_addr\n\nCategory:\nCustomer Data\n\nAliases:\nuser address\ndelivery location\nshipping details\nbilling details\n\nBus

In [21]:
#tokenizer
tokenized_documents = []

import re

def tokenize(text):

    return re.findall(r"\b\w+\b", text.lower())
tokenized_documents = [tokenize(doc) for doc in documents]

<!-- build bm25 -->

In [22]:
tokenized_documents

[['table',
  'customer',
  'category',
  'user',
  'management',
  'aliases',
  'user',
  'buyer',
  'shopper',
  'client',
  'account',
  'holder',
  'member',
  'business',
  'keywords',
  'user',
  'profile',
  'account',
  'contact',
  'email',
  'phone',
  'shopper',
  'member',
  'business',
  'concepts',
  'customer',
  'identity',
  'account',
  'management',
  'user',
  'profile',
  'columns',
  'customer_id',
  'integer',
  'unique',
  'internal',
  'identifier',
  'for',
  'the',
  'registered',
  'user',
  'cust_code',
  'varchar',
  'public',
  'or',
  'external',
  'reference',
  'code',
  'for',
  'the',
  'client',
  'full_name',
  'varchar',
  'complete',
  'legal',
  'or',
  'display',
  'name',
  'of',
  'the',
  'shopper',
  'email',
  'varchar',
  'primary',
  'email',
  'address',
  'used',
  'for',
  'login',
  'and',
  'contact',
  'phone',
  'varchar',
  'primary',
  'mobile',
  'or',
  'landline',
  'number',
  'for',
  'the',
  'user',
  'status',
  'varchar'

In [23]:
from rank_bm25 import BM25Okapi
bm25 = BM25Okapi(tokenized_documents)

In [24]:
import pickle

with open("../data/bm25.pkl","wb") as f:
    pickle.dump(bm25,f)

with open("../data/table_names.pkl","wb") as f:
    pickle.dump(table_names,f)

with open("../data/tokenized_docs.pkl","wb") as f:
    pickle.dump(tokenized_documents,f)

with open("../data/documents.pkl","wb") as f:
    pickle.dump(documents,f)

<!-- search -->

In [25]:
import pickle

with open("../data/bm25.pkl","rb") as f:
    bm25 = pickle.load(f)

with open("../data/table_names.pkl","rb") as f:
    table_names = pickle.load(f)

In [26]:
import numpy as np
def bm25_search(query, top_k=8):

    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked = np.argsort(scores)[::-1]
    results = []
    for rank, idx in enumerate(ranked[:top_k], start=1):
        if (scores[idx]<=0):
            continue
        results.append({
            "table": table_names[idx],
            "rank": rank,
            "score": float(scores[idx]),
            "idx":idx
        })
    return results

In [27]:
print(bm25_search("Find customers who purchased products from different sellers, received shipments from multiple warehouses, returned at least one item, and whose refund has not yet been processed."))

[{'table': 'tbl_ord_item', 'rank': 1, 'score': 10.800286045396163, 'idx': np.int64(8)}, {'table': 'rfnd_log', 'rank': 2, 'score': 10.466300761044998, 'idx': np.int64(12)}, {'table': 'inv_stock', 'rank': 3, 'score': 6.6385616944620764, 'idx': np.int64(5)}, {'table': 'recently_viewed', 'rank': 4, 'score': 2.566502748001246, 'idx': np.int64(17)}, {'table': 'pay_trn', 'rank': 5, 'score': 2.38403262901097, 'idx': np.int64(9)}, {'table': 'wishlist_item', 'rank': 6, 'score': 1.8014083859130552, 'idx': np.int64(16)}, {'table': 'tbl_ord_hdr', 'rank': 7, 'score': 1.3617225567965137, 'idx': np.int64(7)}, {'table': 'coupon', 'rank': 8, 'score': 1.2340090532284997, 'idx': np.int64(13)}]


<!-- embedding models -->


In [28]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [29]:
embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(19, 768)


In [30]:
import numpy as np

np.save(
    "../data/embeddings.npy",
    embeddings
)

In [31]:
import faiss
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [32]:
faiss.write_index(
    index,
    "../data/embedding_index.faiss"
)

<!-- embedding search -->

In [33]:
import faiss
import pickle
from sentence_transformers import SentenceTransformer
index = faiss.read_index(
    "../data/embedding_index.faiss"
)
with open("../data/table_names.pkl","rb") as f:
    table_names = pickle.load(f)

model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [34]:
def embedding_search(query, top_k=8):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    scores, indices = index.search(
        query_embedding,
        top_k
    )
    results = []
    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        results.append({
            "table": table_names[idx],
            "rank": rank,
            "score": float(score),
            "idx":idx
        })
    return results

In [35]:
results = embedding_search(
    "refunded customers"
)
for r in results:
    print(r)

{'table': 'rfnd_log', 'rank': 1, 'score': 0.6473162770271301, 'idx': np.int64(12)}
{'table': 'return_req', 'rank': 2, 'score': 0.6416175365447998, 'idx': np.int64(11)}
{'table': 'pay_trn', 'rank': 3, 'score': 0.5701345205307007, 'idx': np.int64(9)}
{'table': 'tbl_ord_hdr', 'rank': 4, 'score': 0.5680093765258789, 'idx': np.int64(7)}
{'table': 'cart_item', 'rank': 5, 'score': 0.5662773847579956, 'idx': np.int64(15)}
{'table': 'customer', 'rank': 6, 'score': 0.5639280080795288, 'idx': np.int64(0)}
{'table': 'review', 'rank': 7, 'score': 0.560184895992279, 'idx': np.int64(14)}
{'table': 'wishlist_item', 'rank': 8, 'score': 0.5305531024932861, 'idx': np.int64(16)}


<!-- ## RRf -->

In [36]:
from collections import defaultdict

In [37]:
def bm25_search_rrf(query, top_k=8):

    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked = np.argsort(scores)[::-1]
    results = []
    rank = 1
    for idx in ranked:
        if scores[idx] <= 0:
            continue
        results.append({
            "table": table_names[idx],
            "rank": rank,
            "score": float(scores[idx]),
            "idx":idx
        })
        rank += 1
        if len(results) >= top_k:
            break
    return results
def embedding_search_rrf(query, top_k=8):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )
    top_k = min(top_k, len(table_names))
    scores, indices = index.search(query_embedding, top_k)
    results = []
    rank = 1
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append({
            "table": table_names[idx],
            "rank": rank,
            "score": float(score),
            "idx":idx
        })
        rank += 1
    return results

In [38]:
def reciprocal_rank_fusion(
    bm25_results,
    embedding_results,
    embedding_weight=0.8,
    bm25_weight=0.2,
    k=60
):
    fused_scores = defaultdict(float)
    for result in bm25_results:
        fused_scores[result["idx"]] += (
            bm25_weight*(1 / (k + result["rank"]**2))
        )
    for result in embedding_results:
        fused_scores[result["idx"]] += (
            embedding_weight*(1 / (k + result["rank"]**2))
        )
    ranked = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    return ranked

In [39]:
def hybrid_search(query, embedding_weight=0.8,
    bm25_weight=0.2,top_k=7):

    bm25_results = bm25_search_rrf(query, top_k=15)
    embedding_results = embedding_search_rrf(
        query,
        top_k=15
    )
    fused = reciprocal_rank_fusion(
        bm25_results,
        embedding_results,
        embedding_weight,
        bm25_weight
    )
    return fused[:top_k]

In [51]:
query = "products bought after recommendations"
results = hybrid_search(query,embedding_weight=0.9,
    bm25_weight=0.1,top_k=15)
for table in results:
    print(table,table_names[table[0]])

(np.int64(18), 0.01551177536231884) recommendation_log
(np.int64(16), 0.014754098360655738) wishlist_item
(np.int64(17), 0.013043478260869566) recently_viewed
(np.int64(15), 0.011842105263157895) cart_item
(np.int64(8), 0.011014344262295082) tbl_ord_item
(np.int64(14), 0.010588235294117647) review
(np.int64(11), 0.008256880733944955) return_req
(np.int64(4), 0.007258064516129033) product
(np.int64(7), 0.006382978723404255) tbl_ord_hdr
(np.int64(0), 0.005625000000000001) customer
(np.int64(13), 0.005492631004366812) coupon
(np.int64(2), 0.004972375690607734) seller_mst
(np.int64(3), 0.004411764705882353) category
(np.int64(12), 0.003515625) rfnd_log
(np.int64(5), 0.003157894736842105) inv_stock


In [41]:
# bm25_results = bm25_search_rrf(query, top_k=15)
# embedding_results = embedding_search_rrf(
#         query,
#         top_k=15
#     )
# print("BM25")
# for r in bm25_results:
#     print(r)

# print("\nEmbedding")
# for r in embedding_results:
#     print(r)

<!-- ## LLM -->

In [42]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [52]:
candidate_tables = [t for t,_ in results]
print(candidate_tables)

[np.int64(18), np.int64(16), np.int64(17), np.int64(15), np.int64(8), np.int64(14), np.int64(11), np.int64(4), np.int64(7), np.int64(0), np.int64(13), np.int64(2), np.int64(3), np.int64(12), np.int64(5)]


In [53]:
pairs = [(query, documents[t]) for t in candidate_tables]
scores = reranker.predict(pairs)

reranked = sorted(
    zip(candidate_tables, scores),
    key=lambda x: x[1],
    reverse=True
)

In [54]:
for i in reranked:
    print(i,table_names[i[0]])

(np.int64(18), np.float32(-3.3639722)) recommendation_log
(np.int64(8), np.float32(-6.267743)) tbl_ord_item
(np.int64(11), np.float32(-10.160439)) return_req
(np.int64(4), np.float32(-10.510448)) product
(np.int64(7), np.float32(-10.73933)) tbl_ord_hdr
(np.int64(15), np.float32(-10.754747)) cart_item
(np.int64(5), np.float32(-11.127821)) inv_stock
(np.int64(16), np.float32(-11.133387)) wishlist_item
(np.int64(14), np.float32(-11.234216)) review
(np.int64(2), np.float32(-11.264246)) seller_mst
(np.int64(13), np.float32(-11.307968)) coupon
(np.int64(3), np.float32(-11.364939)) category
(np.int64(0), np.float32(-11.376205)) customer
(np.int64(17), np.float32(-11.382227)) recently_viewed
(np.int64(12), np.float32(-11.413894)) rfnd_log


In [46]:
# import sys
# import os
# sys.path.append(os.path.abspath(".."))
# from schema.queries import queries

In [47]:
# import pandas as pd

# rows = []

# for query in queries:

#     # Existing retrieval

#     rrf_results = hybrid_search(query,embedding_weight=0.75,
#     bm25_weight=0.25,top_k=15)

#     # Candidate tables from RRF
#     candidate_tables = [t for t, _ in rrf_results]

#     # Build query-document pairs
#     pairs = [
#         (query,  documents[t])
#         for t in candidate_tables
#     ]

#     # Reranker scores
#     scores = reranker.predict(pairs)

#     # Combine
#     reranked = list(zip(candidate_tables, scores))

#     # Sort descending
#     reranked.sort(
#         key=lambda x: x[1],
#         reverse=True
#     )

#     rows.append({
#         "Query": query,

#         # Original RRF ranking
#         "RRF_Top1": rrf_results[0][0],
#         "RRF_Top5": [t for t, _ in rrf_results[:5]],

#         # After reranking
#         "Reranker_Top1": reranked[0][0],
#         "Reranker_Top5": [t for t, _ in reranked[:5]],

#         # Optional: scores
#         "Scores": [round(s, 4) for _, s in reranked[:5]]
#     })

# df = pd.DataFrame(rows)

# df.to_csv(
#     "reranker_predictions.csv",
#     index=False
# )

# print(df.head())

In [48]:
# df["RRF_Top1"] = df["RRF_Top1"].apply(lambda x: table_names[x])

# df["Reranker_Top1"] = df["Reranker_Top1"].apply(lambda x: table_names[x])

# df["RRF_Top5"] = df["RRF_Top5"].apply(
#     lambda lst: [table_names[i] for i in lst]
# )

# df["Reranker_Top5"] = df["Reranker_Top5"].apply(
#     lambda lst: [table_names[i] for i in lst]
# )

In [49]:
# df.to_csv("reranker_predictions.csv", index=False)

In [50]:
# 